# 03 — Model 2: DenseNet-121 Transfer Learning (CheXNet)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arudaev/chexvision/blob/main/notebooks/03_train_transfer.ipynb)

Fine-tune DenseNet-121 (pretrained on ImageNet) for chest X-ray classification.

> **Cloud-only**: This notebook MUST run on Google Colab (free T4 GPU) or a university cluster. Do NOT run locally.

**Architecture decisions documented here:**
- DenseNet-121 backbone: dense connections maximize feature reuse (Huang et al., 2017)
- ImageNet pretraining: low-level features (edges, textures) transfer well to medical imaging
- Two-phase training:
  - Phase 1: Freeze backbone, train classifier heads only (prevents catastrophic forgetting)
  - Phase 2: Unfreeze all layers with reduced LR (end-to-end fine-tuning)
- CheXNet reference: Rajpurkar et al. (2017) achieved radiologist-level performance with this setup

In [ ]:
# === Colab Setup ===
import os, subprocess, sys

# Verify GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected! Go to Runtime > Change runtime type > T4 GPU. '
        'Training DenseNet-121 on 112k images on CPU is not feasible.'
    )
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# Clone repo and install
if not os.path.exists('/content/chexvision'):
    subprocess.run(['git', 'clone', 'https://github.com/arudaev/chexvision.git', '/content/chexvision'], check=True)
%cd /content/chexvision
!pip install -q -e '.[dev]'
os.environ['CHEXVISION_ALLOW_LOCAL'] = '1'  # We're in Colab, override the guard

In [ ]:
# Download dataset to Colab's ephemeral storage
!python -m src.data.download --output-dir /content/data
print('Dataset ready at /content/data')

In [ ]:
import yaml
from src.models.densenet_transfer import CheXVisionDenseNet

# Load config
with open('configs/transfer.yaml') as f:
    config = yaml.safe_load(f)

config['data']['data_dir'] = '/content/data'
config['logging'] = {'checkpoint_dir': '/content/checkpoints', 'log_dir': '/content/logs'}

# Model info
model = CheXVisionDenseNet(pretrained=True, freeze_backbone=True)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f'Total parameters: {total_params:,}')
print(f'Trainable (Phase 1 — head only): {trainable_params:,}')
print(f'Frozen (Phase 1 — backbone): {frozen_params:,}')
print(f'After Phase 2 unfreeze: all {total_params:,} trainable')
print(f'\nConfig:\n{yaml.dump(config, default_flow_style=False)}')

In [ ]:
# Train (Phase 1: frozen backbone → Phase 2: full fine-tuning)
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

from src.training.trainer import train
train(config)

In [ ]:
# Upload trained model to HuggingFace Hub
from huggingface_hub import HfApi
from google.colab import userdata

api = HfApi(token=userdata.get('HF_TOKEN'))  # Set in Colab Secrets
api.upload_file(
    path_or_fileobj='/content/checkpoints/CheXVision-DenseNet_best.pth',
    path_in_repo='CheXVision-DenseNet_best.pth',
    repo_id='HlexNC/chexvision-densenet',
    repo_type='model',
)
print('Model uploaded to https://huggingface.co/HlexNC/chexvision-densenet')

In [ ]:
# Quick comparison with scratch model (if both checkpoints exist)
from pathlib import Path

scratch_ckpt = Path('/content/checkpoints/CheXVision-ResNet_best.pth')
densenet_ckpt = Path('/content/checkpoints/CheXVision-DenseNet_best.pth')

if scratch_ckpt.exists() and densenet_ckpt.exists():
    !python -m src.training.evaluate --model-dir /content/checkpoints --data-dir /content/data --output-dir /content/results --compare
    from IPython.display import Image, display
    display(Image('/content/results/auc_comparison.png'))
else:
    print('Run notebook 02 first to get the scratch model checkpoint for comparison.')